# Polypeptide Toxicity Classification

Don Frenzel, Rohan Castillo, Lasaro Morell

Binary classification of peptides as **toxic (label 1)** or **non-toxic (label 0)** from
sequence alone. The dataset is 6,387 training and 1,126 test peptides of length 12-51,
split roughly 71% / 29% non-toxic to toxic in both files.

The experiment covers three independent feature representations, each with its own
classifiers, so that the contribution of the representation can be separated from the
contribution of the model:

| Section | Representation | Classifiers |
|---|---|---|
| 1 | Physicochemical property matrix, summarised per sequence | SVM, Random Forest, AdaBoost |
| 2 | Sequence-based: residue sequence | 1D CNN |
| 3 | ProtBert protein language model embeddings | KNN |

Every model is scored with accuracy, sensitivity, specificity and MCC, on 5-fold
cross-validation and on the held-out test set.

**Running this notebook:** `uv run jupyter lab` from the project root, then pick the
project kernel. Paths are relative to the project root, so keep the working directory
there. Run the Setup cell first, then any section top to bottom.

## Setup

Every import used anywhere in the notebook, plus `fastaConverter`, which all three
sections use to read the FASTA files. Run this cell once before any section.

In [ ]:
#----------------IMPORTS----------------#

###Data handling and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

###Biopython, for reading FASTA files
from Bio import SeqIO

###PyTorch, for the CNN in section 2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

###Protein language model, for section 3
from transformers import BertModel, BertTokenizerFast
from tqdm import tqdm

###ML models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

###Model selection, preprocessing and evaluation
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score, confusion_matrix, classification_report, matthews_corrcoef


#----------------FASTA LOADING----------------#

### NOTE LABEL '1' is toxic, LABEL '0' is nontoxic.
# Reads FASTA file and converts to datafram with columns: label, sequence, length
def fastaConverter(filename):
    dataList = []

    with open(filename, "r") as handle:
        for entry in SeqIO.parse(handle, "fasta"):
            dataList.append({
                "label": int(entry.id[8]),
                "sequence": str(entry.seq),
                "length": len(entry.seq)
            })

    return pd.DataFrame(dataList)


---
## 1. Physicochemical features

Each amino acid carries numerical values for physical and chemical properties
(hydrophobicity, charge, polarity and so on), read here from
`data/physiochemical_attributes.xlsx`. A peptide becomes a matrix of
`length x n_properties`, which is then collapsed to a fixed-length vector so that
sequences of different lengths can be fed to the same classifier.

### 1.1 Amino acid property lookup

Builds a dictionary mapping each amino acid letter to its vector of property values.

In [ ]:
# reads amino acid features from an excel_file and returns dictionary mapping amino acid to its features
def amino_acid_features(file_name, sheet_name):
    df = pd.read_excel(file_name, sheet_name, skiprows=1)
    df = df.set_index(df.columns[1])
    df = df.drop(columns=[df.columns[0]])
    df = df.T

    aa_dict = {}

    for aa in df.index:
        aa_dict[aa] = df.loc[aa].values

    return aa_dict

### 1.2 From sequence to fixed-length vector

A sequence becomes a matrix of per-residue property vectors, then that matrix is
summarised by concatenating the mean, standard deviation, minimum, and maximum
for each property, followed by the sequence length.

In [ ]:
# converts a sequence to a matrix: each amino acid -> feature vector
def sequence_to_matrix(sequence, aa_dict):
    return np.array([aa_dict[aa] for aa in sequence])


# converts a matrix to a feature vector containing mean, std, min, and max for each feature across the sequence 
def matrix_to_mean_std_min_max(matrix):
    mean = matrix.mean(axis=0)
    std = matrix.std(axis=0)
    min_val = matrix.min(axis=0)
    max_val = matrix.max(axis=0)

    return np.concatenate([mean, std, min_val, max_val])

In [ ]:
# adds physiochemical feature vectors to the dataframe
def add_physiochemical_features(dataframe, aa_dict):

    # convert each sequence into a matrix
    matrices = dataframe["sequence"].apply(
        lambda seq: sequence_to_matrix(seq, aa_dict)
    )

    dataframe["features"] = matrices.apply(matrix_to_mean_std_min_max)
    dataframe["features"] = dataframe.apply(
        lambda row: np.append(row["features"], row["length"]),
        axis=1
    )

    return dataframe


### 1.3 Hyperparameter search

One grid search per classifier, each using 5-fold stratified cross-validation so every
fold keeps the class ratio of the full set. Hyperparameters are selected by MCC. The SVM is wrapped in a `Pipeline` with
`StandardScaler` so that scaling is fitted inside each fold rather than on the whole
training set, which would leak test-fold statistics into training.

In [ ]:
# does grid search with cross validation for any estimator
def tune_model(estimator, param_grid, X_train, y_train):
    # stratified ensures class balance in fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=44)

    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring=make_scorer(matthews_corrcoef), # metric used to pick best model
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    return grid

In [ ]:
# pipeline to avoid data leakage 
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", random_state=44))
])

search_spaces = {
    "SVM": (
        svm_pipeline,
        {
            "svm__C": [0.1, 1, 5, 10],
            "svm__gamma": [0.001, 0.01, 0.05, 0.1],
            "svm__class_weight": [None, "balanced"]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=44),
        {
            "n_estimators": [10, 50, 100, 200, 400],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4]
        }
    ),
    "AdaBoost": (
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(random_state=44),
            random_state=44
        ),
        {
            "n_estimators": [10, 50, 100, 200, 400],
            "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
            "estimator__max_depth": [1, 2, 3],
            "estimator__min_samples_leaf": [1, 2, 4]
        }
    ),
}

### 1.4 Scoring

Refits the chosen model on the full training set and reports the four metrics on the
independent test set. Sensitivity is recall on the toxic class, specificity is recall on
the non-toxic class, and MCC is the metric to watch given the class imbalance.

In [ ]:
# models arrive already fitted: GridSearchCV refits the best estimator on the
# full training set
def evaluate_model(X_test, y_test, model):
    y_pred = model.predict(X_test)

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    TN, FP, FN, TP = cm.ravel()

    accuracy = accuracy_score(y_test, y_pred)
    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)
    mcc = matthews_corrcoef(y_test, y_pred)

    return cm, accuracy, sensitivity, specificity, mcc

### 1.5 Run: load and featurise

Uses the mean, standard deviation, minimum, maximum, and sequence length.

In [ ]:
aa_dict = amino_acid_features("data/physiochemical_attributes.xlsx", "Sheet2")

# load training and test data
trainingData = fastaConverter("data/Train.fasta")
testingData = fastaConverter("data/test.fasta")


# convert sequences to features
trainingData = add_physiochemical_features(
    trainingData,
    aa_dict
)
testingData = add_physiochemical_features(
    testingData,
    aa_dict
)

# convert to numpy arrays
X_train = np.vstack(trainingData["features"].values)
y_train = trainingData["label"].values

X_test = np.vstack(testingData["features"].values)
y_test = testingData["label"].values

### 1.6 Run: grid search

In [ ]:
grids = {
    name: tune_model(estimator, param_grid, X_train, y_train)
    for name, (estimator, param_grid) in search_spaces.items()
}

# print best parameters and cv performance
for name, grid in grids.items():
    print(f"Best {name} Params:")
    print(grid.best_params_)
    print(f"Best {name} CV MCC: {grid.best_score_:.4f}\n")

### 1.7 Run: test set results

In [ ]:
# evaluate all models
for name, grid in grids.items():
    cm, accuracy, sensitivity, specificity, mcc = evaluate_model(
        X_test, y_test, grid.best_estimator_)

    print(f"\nFinal {name} Test Results")
    print("Confusion Matrix:")
    print(cm)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"MCC: {mcc:.4f}")

### 1.8 Recorded results

From `results/physiochemical_features__classifier_results.txt` (previous accuracy-selected
run; rerun the searches and evaluation to obtain MCC-selected results):

| Model | CV accuracy | Test accuracy | Sensitivity | Specificity | MCC |
|---|---|---|---|---|---|
| SVM | 0.9032 | 0.8970 | 0.7625 | 0.9504 | 0.7404 |
| Random Forest | 0.9061 | 0.9005 | 0.7781 | 0.9491 | 0.7501 |
| AdaBoost | 0.9023 | 0.9041 | 0.8125 | 0.9404 | 0.7618 |

All three land near 0.90 accuracy, and all three are far better at recognising
non-toxic peptides than toxic ones: specificity sits around 0.94-0.95 while sensitivity
reaches only 0.76-0.81. These recorded grid searches selected on accuracy, which on a
71/29 split rewards the majority class.

---
## 2. Sequence-based features

This section uses only the ordering of the letters, with no reference to the chemical
nature of the amino acids. The input is the peptide itself, one integer index per amino
acid, which the CNN turns into learned embeddings.

That is the reason a CNN is appropriate here. Convolution assumes that neighbouring
inputs are related, and along a peptide they are: adjacent residues are physically
adjacent. A kernel sliding along the sequence acts as a motif detector, a learned
generalisation of bigram counts.

An earlier version ran a 2D CNN over the 20x20 bigram matrix instead. That matrix's rows
and columns are in alphabetical order, so neighbouring cells are not related and 2D
convolution has no real structure to exploit.

### 2.1 The CNN

Each residue index is mapped to a learned 32-dimensional embedding. One 1D convolution
with 64 filters of width 5 slides along the sequence, so each filter responds to a
5-residue pattern. Global max pooling keeps each filter's strongest response anywhere in
the peptide, which handles variable lengths and finds a motif wherever it occurs. Dropout
and one linear layer give the two class scores.

Padding positions are set to `-inf` before the max pool, so a short peptide is never
scored on the empty space after it.

In [ ]:
class PeptideCNN(nn.Module):
    def __init__(self, embedding_dim=32, num_filters=64, kernel_size=5):
        super().__init__()
        # 21 = 20 amino acids + index 0 for padding
        self.embedding = nn.Embedding(21, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size, padding="same")
        self.dropout = nn.Dropout(0.5)
        self.out = nn.Linear(num_filters, 2)

    def forward(self, x):
        is_padding = x == 0                       # (batch, length)
        x = self.embedding(x)                     # (batch, length, embedding_dim)
        x = x.permute(0, 2, 1)                    # Conv1d wants (batch, channels, length)
        x = F.relu(self.conv(x))                  # (batch, num_filters, length)
        x = x.masked_fill(is_padding.unsqueeze(1), float("-inf"))
        x = x.max(dim=2).values                   # (batch, num_filters)
        return self.out(self.dropout(x))

### 2.2 Device and data

Parsing is done with Biopython's `SeqIO` rather than a plain text split, because the
earlier fastaframes approach did not separate the columns correctly.


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Running on: ',device)

In [ ]:
###Load data in, training/testing.  Split later for validation.  
trainingData = fastaConverter('data/Train.fasta')
testingData = fastaConverter('data/test.fasta')

###Important for functions, essentially acts as a library for amino acids.  
allColumns = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']


### 2.3 Sequence encoding

Each amino acid becomes an integer from 1 to 20, and every peptide is right-padded with 0
to the length of the longest one so they stack into a single tensor. Because global max
pooling removes the length dimension, train and test can be padded to different lengths.

In [ ]:
aa_to_index = {aa: i + 1 for i, aa in enumerate(allColumns)}  # 0 is reserved for padding


def encode_sequences(sequences):
    """Turns each peptide into a row of amino acid indices, right-padded with 0."""
    max_length = max(len(sequence) for sequence in sequences)
    encoded = np.zeros((len(sequences), max_length), dtype=np.int64)
    for row, sequence in enumerate(sequences):
        encoded[row, : len(sequence)] = [aa_to_index[aa] for aa in sequence]
    return torch.from_numpy(encoded)

### 2.4 CNN training and evaluation

Holds out a stratified 20% of the training set for validation, trains with cross-entropy
loss and Adam, and prints the validation MCC after each epoch. The test set is used once,
at the end, and scored with the same four metrics as Section 1.

In [ ]:
def train_one_epoch(model, optimizer, x_train, y_train, batch_size=32):
    loss_function = nn.CrossEntropyLoss()
    model.train()
    order = torch.randperm(len(x_train))
    total_loss = 0.0
    for start in range(0, len(x_train), batch_size):
        batch = order[start : start + batch_size]
        optimizer.zero_grad()
        loss = loss_function(model(x_train[batch]), y_train[batch])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)
    return total_loss / len(x_train)


def predict_labels(model, x):
    model.eval()
    with torch.no_grad():
        return model(x).argmax(dim=1).cpu().numpy()


def print_test_results(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = cm.ravel()
    mcc = matthews_corrcoef(y_test, y_pred)

    print("\nFinal CNN Test Results")
    print("Confusion Matrix:")
    print(cm)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Sensitivity: {TP / (TP + FN):.4f}")
    print(f"Specificity: {TN / (TN + FP):.4f}")
    print(f"MCC: {mcc:.4f}")
    return mcc


def plot_validation_curve(val_mcc_per_epoch, test_mcc):
    plt.plot(range(1, len(val_mcc_per_epoch) + 1), val_mcc_per_epoch)
    plt.axhline(y=test_mcc, color='gold', linestyle='--')
    plt.title('Model MCC, 1D CNN')
    plt.ylabel('MCC')
    plt.xlabel('Epochs')
    plt.legend(['Val', 'Test'], loc='lower right')
    plt.show()


def run_cnn(trainingData, testingData, num_epochs):
    torch.manual_seed(0)
    train_rows, val_rows = train_test_split(
        trainingData, test_size=0.2, stratify=trainingData.label, random_state=0)
    x_train = encode_sequences(train_rows.sequence).to(device)
    y_train = torch.from_numpy(train_rows.label.to_numpy()).to(device)
    x_val = encode_sequences(val_rows.sequence).to(device)
    x_test = encode_sequences(testingData.sequence).to(device)

    model = PeptideCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    val_mcc_per_epoch = []
    for epoch in range(num_epochs):
        loss = train_one_epoch(model, optimizer, x_train, y_train)
        val_mcc = matthews_corrcoef(val_rows.label, predict_labels(model, x_val))
        val_mcc_per_epoch.append(val_mcc)
        print(f"Epoch {epoch + 1}: train loss {loss:.3f}, validation MCC {val_mcc:.3f}")

    test_mcc = print_test_results(testingData.label, predict_labels(model, x_test))
    plot_validation_curve(val_mcc_per_epoch, test_mcc)

### 2.5 Run

In [ ]:
run_cnn(trainingData, testingData, 30)

### 2.6 Recorded results

From a 30-epoch run of `run_cnn` (seed 0):

| Model | Features | Test accuracy | Sensitivity | Specificity | MCC |
|---|---|---|---|---|---|
| 1D CNN | Residue sequence | 0.9227 | 0.8187 | 0.9640 | 0.8065 |

The 1D CNN is the strongest model in the whole experiment.

---
## 3. Protein language model embeddings

A protein language model is trained on large protein databases and produces a numerical
encoding of a sequence that can be used directly as a feature vector. This section uses
`Rostlab/prot_bert`, a BERT model trained on protein sequences, and classifies its
embeddings with KNN.

Unlike the previous two sections, no features are hand-designed here: the representation
is entirely learned, so this is the test of whether a pretrained model beats explicit
feature engineering on this task.

**Note:** this section downloads model weights on first run (a few hundred MB) and will
use a GPU if one is available.

### 3.1 Reading the data

In [ ]:
trainingData = fastaConverter('data/Train.fasta')
testingData = fastaConverter('data/test.fasta')

### 3.2 Loading ProtBert and extracting embeddings

ProtBert on HuggingFace expects residues separated by spaces, so `ACDEF` has to be
rewritten as `A C D E F`. Sequences are processed in batches, and the hidden state at
index 0, the `[CLS]` token, is taken as the summary vector for the whole peptide.

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

In [ ]:
def proteinbert_feature_extractor(peptides, batch_size=32):
      features = []
      for i in tqdm(range(0, len(peptides), batch_size)):
        batch_seqs = peptides[i : i + batch_size]
        
        # Format: "ACDEF" -> "A C D E F", necessary for pbert huggingface
        spaced_seqs = [" ".join(list(seq)) for seq in batch_seqs]
        
        # Tokenize
        inputs = tokenizer(spaced_seqs, return_tensors='pt', padding=True, truncation=True, max_length=52)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Use the index 0 as summary
            batch_features = outputs.last_hidden_state[:, 0, :].cpu()
            features.append(batch_features)
            
      return torch.cat(features, dim=0)

### 3.3 KNN on the embeddings

Grid searches the neighbour count with 5-fold cross-validation, using distance weighting
so that closer neighbours count for more.

In [ ]:
def knn_plm(data):
    peptide_features = proteinbert_feature_extractor(data['sequence'])
    pep_labels = torch.tensor(data['label'].values).float().unsqueeze(1)
    X = peptide_features.numpy() 
    y = data['label'].values

    # 2. Split into training and testing (80/20 split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. K-fold validation KNN
    grid_search = GridSearchCV(KNeighborsClassifier(weights='distance'), {'n_neighbors': range(1, (int(6387**(1/2))))}, cv=5)
    grid_search.fit(X_train, y_train)

    # 4. Evaluate
    y_pred = grid_search.predict(X_test)
    print(f"Best k param: {grid_search.best_params_} best score: {grid_search.best_score_}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(classification_report(y_test, y_pred))
    print(f"Matthews correlation coefficient(MCC): {matthews_corrcoef(y_test, y_pred)}")

### 3.4 Run

In [ ]:
knn_plm(trainingData)

### 3.5 Results

No saved output for this section yet.

---
## 4. Discussion

Collecting every recorded test-set number in one place:

| Section | Representation | Model | Accuracy | Sensitivity | Specificity | MCC |
|---|---|---|---|---|---|---|
| 2 | Residue sequence | 1D CNN | 0.9227 | 0.8187 | 0.9640 | 0.8065 |
| 1 | Physicochemical | AdaBoost | 0.9041 | 0.8125 | 0.9404 | 0.7618 |
| 1 | Physicochemical | Random Forest | 0.9005 | 0.7781 | 0.9491 | 0.7501 |
| 1 | Physicochemical | SVM | 0.8970 | 0.7625 | 0.9504 | 0.7404 |
| 3 | ProtBert | KNN | - | - | - | - |

Points worth developing in the report:

1. **Physicochemical features are close behind with much less machinery.** Three
   conventional classifiers all reach roughly 0.90 on a vector of summary statistics,
   within a couple of points of the CNN.

2. **Every model is weaker on the toxic class.** Sensitivity trails specificity by 13 to
   18 points throughout. With a 71/29 split and grid searches that select on accuracy,
   this is the expected outcome and is worth addressing directly rather than reporting
   as a curiosity.

3. **The PLM section is not yet comparable.** It evaluates on an 80/20 split of the
   training data rather than on the independent test set, so its numbers cannot be put
   in the table above as they stand.